In [7]:
import pandas as pd
import numpy as np

clean = pd.read_parquet('../data/clean_entries.parquet')

split = pd.Timestamp('2025-09-17')
train = clean[(clean['Date'] < split) & (~clean['is_holiday'])].copy()
test = clean[clean['Date'] >= split].copy()
print('train days:', train['Date'].unique(), "| test days:", test['Date'].nunique())
print('test day types:', test.drop_duplicates('Date')['day_type'].value_counts().to_dict())

peak = train.groupby('Station')['Ridership'].quantile(0.95).clip(lower=1).rename('peak')
train = train.merge(peak, on='Station')
test = test.merge(peak, on='Station')
for d in (train, test):
    d['crowd'] = (d['Ridership'] / d['peak']).clip(0, 1)

b0 = train.groupby(['Station', 'Hour'])['crowd'].mean().rename('pred_b0')

b1 = train.groupby(['Station', 'day_type', 'Hour'])['crowd'].mean().rename('pred_b1')
test = test.join(b0, on=['Station', 'Hour']).join(b1, on=['Station', 'day_type', 'Hour'])
print('missing predictions:', test[['pred_b0', 'pred_b1']].isna().sum().to_dict())

def report(label, mask):
    t = test[mask]
    print(f'{label:28s} baseline0 MAE={np.abs(t.crowd - t.pred_b0).mean():.4f} |'
          f' baseline1 MAE={np.abs(t.crowd - t.pred_b1).mean():.4f}')

report('all hours', test['Hour'] >= 0)
report('service hours (6 to 22)', test['Hour'].between(6, 22))
for dt in ['weekday', 'saturday', 'sunday']:
    report(f'service hours, {dt}', test['Hour'].between(6, 22) & (test['day_type'] == dt))

train days: <DatetimeArray>
['2025-08-11 00:00:00', '2025-08-12 00:00:00', '2025-08-13 00:00:00',
 '2025-08-14 00:00:00', '2025-08-16 00:00:00', '2025-08-17 00:00:00',
 '2025-08-18 00:00:00', '2025-09-01 00:00:00', '2025-09-02 00:00:00',
 '2025-09-03 00:00:00', '2025-09-04 00:00:00', '2025-09-05 00:00:00',
 '2025-09-06 00:00:00', '2025-09-07 00:00:00', '2025-09-08 00:00:00',
 '2025-09-09 00:00:00', '2025-09-10 00:00:00', '2025-09-11 00:00:00',
 '2025-09-12 00:00:00', '2025-09-13 00:00:00', '2025-09-14 00:00:00',
 '2025-09-15 00:00:00', '2025-09-16 00:00:00']
Length: 23, dtype: datetime64[us] | test days: 14
test day types: {'weekday': 10, 'saturday': 2, 'sunday': 2}
missing predictions: {'pred_b0': 0, 'pred_b1': 0}
all hours                    baseline0 MAE=0.0645 | baseline1 MAE=0.0352
service hours (6 to 22)      baseline0 MAE=0.0894 | baseline1 MAE=0.0483
service hours, weekday       baseline0 MAE=0.0696 | baseline1 MAE=0.0472
service hours, saturday      baseline0 MAE=0.1131 | base

In [9]:
import time
from sklearn.ensemble import HistGradientBoostingRegressor
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor

st_code = {s: i for i, s in enumerate(sorted(clean['Station'].unique()))}

def make_X(d):
    return pd.DataFrame({'station': d['Station'].map(st_code).values,
                         'hour':d['Hour'].values,
                         'dow': d['dow'].values})
cats = pd.CategoricalDtype(sorted(st_code.values()))
def X_cat(d):
    X = make_X(d)
    X["station"] = X["station"].astype(cats)
    return X

models = {
    "HistGradientBoosting": (HistGradientBoostingRegressor(categorical_features=[0], max_iter=300,
                                                           learning_rate=0.1, random_state=42),
                             make_X(train), make_X(test)),
    "LightGBM": (LGBMRegressor(n_estimators=300, learning_rate=0.1, random_state=42, verbose=-1),
                 X_cat(train), X_cat(test)),
    "XGBoost": (XGBRegressor(n_estimators=300, learning_rate=0.1, tree_method="hist",
                             enable_categorical=True, random_state=42),
                X_cat(train), X_cat(test)),
}

fit_time = {}
for name, (m, Xtr, Xte) in models.items():
    t0 = time.time()
    m.fit(Xtr, train["crowd"])
    fit_time[name] = time.time() - t0
    test["pred_" + name] = np.clip(m.predict(Xte), 0, 1)

preds = {"Baseline (avg table)": "pred_b1", **{n: "pred_" + n for n in models}}
svc = test["Hour"].between(6, 22)
slices = {"service, all": svc,
          "weekday": svc & (test["day_type"] == "weekday"),
          "saturday": svc & (test["day_type"] == "saturday"),
          "sunday": svc & (test["day_type"] == "sunday")}

mae = pd.DataFrame({sl: {n: np.abs(test.loc[mk, "crowd"] - test.loc[mk, c]).mean()
                         for n, c in preds.items()} for sl, mk in slices.items()})
print("MAE (service hours 6 to 22, lower is better):")
print(mae.round(4).to_string())

summ = pd.DataFrame({
    "RMSE": {n: np.sqrt(((test.loc[svc, "crowd"] - test.loc[svc, c]) ** 2).mean()) for n, c in preds.items()},
    "bias": {n: (test.loc[svc, c] - test.loc[svc, "crowd"]).mean() for n, c in preds.items()},
    "fit_sec": {n: fit_time.get(n, 0.0) for n in preds},
})
print("\nService hours, all days:")
print(summ.round(4).to_string())

MAE (service hours 6 to 22, lower is better):
                      service, all  weekday  saturday  sunday
Baseline (avg table)        0.0483   0.0472    0.0559  0.0463
HistGradientBoosting        0.0515   0.0511    0.0569  0.0482
LightGBM                    0.0514   0.0511    0.0565  0.0481
XGBoost                     0.0507   0.0505    0.0558  0.0468

Service hours, all days:
                        RMSE    bias  fit_sec
Baseline (avg table)  0.0748 -0.0096   0.0000
HistGradientBoosting  0.0785 -0.0100   0.9015
LightGBM              0.0783 -0.0103   0.2633
XGBoost               0.0782 -0.0103   0.4520


In [11]:
dt_code = {'weekday':0, 'saturday': 1, 'sunday': 2}

def X_dt(d):
    X = pd.DataFrame({"station": d["Station"].map(st_code).values,
                      "hour": d["Hour"].values,
                      "day_type": d["day_type"].map(dt_code).values})
    X["station"] = X["station"].astype(cats)
    return X

lgb2 = LGBMRegressor(n_estimators=200, learning_rate=0.05, min_child_samples=20,
                     random_state=42, verbose=-1)
lgb2.fit(X_dt(train), train["crowd"])
test["pred_lgb_dt"] = np.clip(lgb2.predict(X_dt(test)), 0, 1)
test["pred_blend"] = (test["pred_b1"] + test["pred_lgb_dt"]) / 2   # average of baseline and model

cols = {"Baseline": "pred_b1", "LightGBM (day_type)": "pred_lgb_dt", "Blend (baseline + LGBM)": "pred_blend"}
svc = test["Hour"].between(6, 22)
sl = {"service, all": svc, "weekday": svc & (test["day_type"] == "weekday"),
      "saturday": svc & (test["day_type"] == "saturday"), "sunday": svc & (test["day_type"] == "sunday")}
out = pd.DataFrame({k: {n: np.abs(test.loc[m, "crowd"] - test.loc[m, c]).mean() for n, c in cols.items()}
                    for k, m in sl.items()})
print(out.round(4).to_string())

                         service, all  weekday  saturday  sunday
Baseline                       0.0483   0.0472    0.0559  0.0463
LightGBM (day_type)            0.0535   0.0523    0.0623  0.0507
Blend (baseline + LGBM)        0.0493   0.0486    0.0554  0.0464


In [15]:
import pickle

full = clean[~clean["is_holiday"]].copy()
peak_f = full.groupby("Station")["Ridership"].quantile(0.95).clip(lower=1)
full["crowd"] = (full["Ridership"] / full["Station"].map(peak_f)).clip(0, 1)
table = full.groupby(["Station", "day_type", "Hour"])["crowd"].mean()

with open('../models/crowd_model.pkl', 'wb') as f:
    pickle.dump({
        'table': table.to_dict(),
        'stations': sorted(peak_f.index),
        "trained_on": f"{full['Date'].min().date()} to {full['Date'].max().date()}",
        "heldout_mae_service_hours": 0.0483,
    }, f)

with open("../models/crowd_model.pkl", "rb") as f:
    m = pickle.load(f)


def predict_crowd(station, day_type, hour):
    return m['table'][(station, day_type, hour)]

st = 'Nadaprabhu Kempegowda Station, Majestic'
for dt, h in [('weekday', 3), ('weekday', 9), ('weekday', 13), ('weekday', 18), ('sunday', 13)]:
    print(f'{st} | {dt:8s} | {h:2d}:00 -> {predict_crowd(st, dt, h):.0%}')
print('entries in table:', len(m['table']), "| stations:", len(m['stations']))

Nadaprabhu Kempegowda Station, Majestic | weekday  |  3:00 -> 0%
Nadaprabhu Kempegowda Station, Majestic | weekday  |  9:00 -> 89%
Nadaprabhu Kempegowda Station, Majestic | weekday  | 13:00 -> 60%
Nadaprabhu Kempegowda Station, Majestic | weekday  | 18:00 -> 94%
Nadaprabhu Kempegowda Station, Majestic | sunday   | 13:00 -> 90%
entries in table: 5976 | stations: 83
